In [7]:
# [셀 1] 라이브러리 로드 및 크롬 브라우저 실행
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd

# 크롬 옵션 설정 (창이 보이도록 설정)
options = webdriver.ChromeOptions()
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
# options.add_argument('--headless') # 창을 보기 위해 주석 처리 또는 삭제

# 드라이버 실행
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
driver.maximize_window() # 화면이 작아서 요소가 숨는 것을 방지
print("🚀 크롬 브라우저가 정상적으로 시작되었습니다.")

# [셀 2에 있던 거]성별=여성(gf=F) 및 유형=무신사 스냅(types=CODISHOP_SNAP) 파라미터 반영
target_url = "https://www.musinsa.com/app/styles/lists?types=CODISHOP_SNAP&gf=F"
print(f"🚀 무신사 여성 스냅 페이지로 이동합니다: {target_url}")
driver.get(target_url)

print("\n💡 [안내] 크롬 창에서 직접 원하는 '스타일 필터(미니멀, 록시크, Y2K 등)'를 마우스로 클릭하세요.")
print("💡 필터를 클릭하고 페이지가 완전히 새로고침된 것을 확인한 후, 아래 [셀 3]을 실행하시면 됩니다.")

🚀 크롬 브라우저가 정상적으로 시작되었습니다.
🚀 무신사 여성 스냅 페이지로 이동합니다: https://www.musinsa.com/app/styles/lists?types=CODISHOP_SNAP&gf=F

💡 [안내] 크롬 창에서 직접 원하는 '스타일 필터(미니멀, 록시크, Y2K 등)'를 마우스로 클릭하세요.
💡 필터를 클릭하고 페이지가 완전히 새로고침된 것을 확인한 후, 아래 [셀 3]을 실행하시면 됩니다.


In [8]:
# [셀 2] 🛠️ [1단계] 스냅 내 상품 ID 전용 수집 및 CSV 저장 코드
import re
import time
import os
import pandas as pd
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# =====================================================================
# ✍️ [설정] 원하는 스타일 태그명 지정
# =====================================================================
SELECTED_STYLE = "클래식+미니멀+프레피"  
TARGET_SNAP_COUNT = 100   # 목표 스냅 개수
output_csv = f"musinsa_goods_nos_{SELECTED_STYLE}_ordered.csv"

print(f"=== [1단계] 무신사 여성 스냅 -> [{SELECTED_STYLE}] 고속 ID 추출 수집 ===")

# --- 1. [가상 스크롤 타파] 100개 스냅 링크가 모일 때까지 무조건 스크롤 다운 ---
snap_links = []
print("🔄 가상 스크롤을 우회하여 100개의 스냅 주소를 수집 중입니다...")

# 최대 50번 스크롤 튕기기 시도
for i in range(50): 
    # 화면에 노출된 스냅 카드 카드 추출
    cards = driver.find_elements(By.CSS_SELECTOR, "[class*='RecoFeedList__FeedCard'], [class*='SnapFeedCard__Container']")
    for card in cards:
        try:
            a_tag = card.find_element(By.TAG_NAME, "a")
            link = a_tag.get_attribute("href")
            if link and "/snap/" in link:
                snap_links.append(link)
        except:
            continue
            
    # 중복 제거
    snap_links = list(dict.fromkeys(snap_links))
    print(f"  └ 현재까지 확보된 스냅 링크: {len(snap_links)}개 / 목표: {TARGET_SNAP_COUNT}개")
    
    if len(snap_links) >= TARGET_SNAP_COUNT:
        print("✨ 목표치인 100개 이상의 스냅 링크를 확보했습니다!")
        break
        
    # PAGE_DOWN 키 연사를 통해 가상 스크롤 강제 트리거
    driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.PAGE_DOWN)
    time.sleep(1.2)

# 최종 타겟 슬라이싱
test_targets = snap_links[:TARGET_SNAP_COUNT]
collected_goods_dict = {}

# --- 2. 각 스냅 페이지로 진입하여 상품 고유 ID 추출 (0개 방지 동적 대기 추가) ---
if test_targets:
    print(f"\n🎯 확보된 {len(test_targets)}개 스냅 순회 및 상품 ID 박제를 시작합니다.")
    
    for idx, target_snap in enumerate(test_targets, start=1):
        try:
            print(f"📸 [스냅 이동 {idx}/{len(test_targets)}] {target_snap}")
            
            # 가상 돔 찌꺼기 꼬임 방지를 위해 완벽 리로드 이동
            driver.get(target_snap)
            
            # 🔥 [0개 방지 치트키] 무신사 내부 데이터(div[data-item-id])가 렌더링 완료될 때까지 동적 대기
            try:
                WebDriverWait(driver, 7).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "div[data-item-id]"))
                )
                time.sleep(0.8) # 미세 안착 버퍼
            except:
                print("   ⚠️ 상품 태그 로딩 지연... 백업 파싱을 진행합니다.")
                
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            
            # 정석 선택자 서치
            product_elements = soup.select("div[data-item-id]")
            if not product_elements:
                product_elements = soup.select("div[class*='SnapFeedProductWrapper__Container']")
                
            print(f"   └ 📦 태그된 상품 개수: {len(product_elements)}개 수집 성공")
            
            for product in product_elements:
                product_id = product.get('data-item-id')
                item_index = product.get('data-item-list-index')
                
                if product_id and product_id.isdigit():
                    if product_id not in collected_goods_dict:
                        idx_val = int(item_index) if (item_index and item_index.isdigit()) else len(collected_goods_dict)
                        collected_goods_dict[product_id] = {
                            "display_order": idx_val,
                            "snap_url": target_snap
                        }
                        
        except Exception as e:
            print(f"❌ {idx}번째 스냅 처리 중 오류 발생: {e}")
            continue
else:
    print("❌ 목록에서 스냅 링크를 단 하나도 찾지 못했습니다.")

# --- 3. 최종 데이터셋 정렬 및 저장 ---
if collected_goods_dict:
    records = []
    for p_id, info in collected_goods_dict.items():
        records.append({
            "goodsNo": p_id,
            "style_tag": SELECTED_STYLE,
            "snap_url": info["snap_url"],
            "display_order": info["display_order"]
        })
    df = pd.DataFrame(records).sort_values(by="display_order").reset_index(drop=True)
    df_final = df[["goodsNo", "style_tag", "snap_url"]]
    df_final.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"\n🎉 [{SELECTED_STYLE}] 스타일 주소록 생성 완료: '{output_csv}' (총 {len(df_final)}개 상품 번호)")

=== [1단계] 무신사 여성 스냅 -> [클래식+미니멀+프레피] 고속 ID 추출 수집 ===
🔄 가상 스크롤을 우회하여 100개의 스냅 주소를 수집 중입니다...
  └ 현재까지 확보된 스냅 링크: 21개 / 목표: 100개
  └ 현재까지 확보된 스냅 링크: 30개 / 목표: 100개
  └ 현재까지 확보된 스냅 링크: 36개 / 목표: 100개
  └ 현재까지 확보된 스냅 링크: 45개 / 목표: 100개
  └ 현재까지 확보된 스냅 링크: 51개 / 목표: 100개
  └ 현재까지 확보된 스냅 링크: 60개 / 목표: 100개
  └ 현재까지 확보된 스냅 링크: 66개 / 목표: 100개
  └ 현재까지 확보된 스냅 링크: 75개 / 목표: 100개
  └ 현재까지 확보된 스냅 링크: 84개 / 목표: 100개
  └ 현재까지 확보된 스냅 링크: 90개 / 목표: 100개
  └ 현재까지 확보된 스냅 링크: 99개 / 목표: 100개
  └ 현재까지 확보된 스냅 링크: 105개 / 목표: 100개
✨ 목표치인 100개 이상의 스냅 링크를 확보했습니다!

🎯 확보된 100개 스냅 순회 및 상품 ID 박제를 시작합니다.
📸 [스냅 이동 1/100] https://www.musinsa.com/snap/1514148237458964228
   └ 📦 태그된 상품 개수: 12개 수집 성공
📸 [스냅 이동 2/100] https://www.musinsa.com/snap/1514148237886781206
   └ 📦 태그된 상품 개수: 13개 수집 성공
📸 [스냅 이동 3/100] https://www.musinsa.com/snap/1508777076124687366
   └ 📦 태그된 상품 개수: 10개 수집 성공
📸 [스냅 이동 4/100] https://www.musinsa.com/snap/1508777078129562590
   └ 📦 태그된 상품 개수: 14개 수집 성공
📸 [스냅 이동 5/100] https://www.musinsa.com/snap/15

In [3]:
import re
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By

print("🚀 현재 화면에 노출된 스냅 카드의 상세 페이지 링크 수집을 시작합니다.")

# 1. 무신사 스냅 카드 요소 탐색 (class명 내부 패턴 매칭)
snap_cards = driver.find_elements(By.CSS_SELECTOR, "[class*='RecoFeedList__FeedCard'], [class*='SnapFeedCard__Container']")
snap_links = []

# 2. 각 스냅 카드 내부에서 하이퍼링크(href) 추출
for card in snap_cards:
    try:
        a_tag = card.find_element(By.TAG_NAME, "a")
        link = a_tag.get_attribute("href")
        if link and "/snap/" in link:
            snap_links.append(link)
    except:
        continue

# 중복 제거 및 리스트 정제
snap_links = list(dict.fromkeys(snap_links))
print(f"✅ 총 {len(snap_links)}개의 스냅 상세 페이지 링크를 발견했습니다.")

# --- 데이터 누적 저장할 리스트 생성 ---
collected_brand_data = []

# 3. 발견한 스냅 링크로 순차 이동 및 카테고리까지 깊게(Deep) 수집 (상위 10개 우선 테스트)
if snap_links:
    print("\n🎯 상위 10개 스냅 페이지 진입 및 하위 상품 카테고리 데이터 수집을 시작합니다.")
    test_targets = snap_links[:10]
    
    for idx, target_snap in enumerate(test_targets, start=1):
        try:
            print(f"\n==================================================")
            print(f"📸 [스냅 이동 {idx}/{len(test_targets)}] {target_snap}")
            driver.get(target_snap)
            time.sleep(2.5) # 동적 데이터 로딩 대기
            
            # BeautifulSoup으로 현재 스냅 상세 페이지 파싱
            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')
            
            # 제공해주신 div 태그의 data-item-id 속성을 기준으로 상품 컨테이너 탐색
            product_elements = soup.select("div[data-item-id]")
            print(f"-> 태그된 상품 개수: {len(product_elements)}개 발견")
            
            # 💡 브라우저가 이동하면 기존 element 정보가 날아가므로, 루프 돌릴 상품 ID 및 기본 메타 정보를 먼저 메모리에 박제
            snap_products = []
            for product in product_elements:
                brand_name = product.get('data-item-brand')
                item_index = product.get('data-item-list-index')
                product_id = product.get('data-item-id')
                if product_id:
                    snap_products.append({
                        'product_id': product_id,
                        'item_index': item_index,
                        'brand_name': brand_name
                    })
            
            # 4. 각 상품별 페이지로 2차 진입하여 카테고리 수집
            for p_info in snap_products:
                p_id = p_info['product_id']
                product_url = f"https://www.musinsa.com/products/{p_id}"
                category_name = "미분류"  # 기본값 설정
                
                try:
                    print(f"   🛒 [상품 진입] ID: {p_id} -> {product_url}")
                    driver.get(product_url)
                    time.sleep(2.0)  # 상품 페이지 로딩 대기 및 서버 부하 방지
                    
                    # 상품 페이지 파싱
                    p_html = driver.page_source
                    p_soup = BeautifulSoup(p_html, 'html.parser')
                    
                    # 제공해주신 앵커 태그 패턴 반영: data-button-name="상품카테고리" 탐색
                    cate_tag = p_soup.select_one('a[data-button-name="상품카테고리"]')
                    
                    if cate_tag and cate_tag.get('data-category-name'):
                        category_name = cate_tag.get('data-category-name')
                    elif cate_tag:
                        # 백업: 속성이 비어있다면 태그 텍스트(상의, 하의 등) 직접 추출
                        category_name = cate_tag.get_text(strip=True)
                        
                    print(f"      ㄴ 🏷️ 카테고리 추출 완료: {category_name} | 브랜드: {p_info['brand_name']}")
                    
                except Exception as p_e:
                    print(f"      🔺 상품 페이지 처리 중 오류 (스킵): {p_e}")
                
                # 수집된 데이터를 통합 리스트에 저장
                collected_brand_data.append({
                    'snap_url': target_snap,
                    'item_index': p_info['item_index'],
                    'product_id': p_id,
                    'brand_name': p_info['brand_name'],
                    'category_name': category_name  # ✨ 카테고리 추가
                })
            
            # ⚠️ [중요 요건] 스냅 내 모든 상품을 순회했으면 다음 랭크 스냅으로 넘어가기 위해 목록에서 정상 리로드 처리
            # (사실 다음 루프 시작 시 driver.get(target_snap)으로 바로 이동하므로 자연스럽게 해결됩니다.)
            
        except Exception as e:
            print(f"❌ {idx}번째 스냅 처리 중 오류 발생: {e}")
            
    print("\n💡 뎁스(Depth) 수집이 모두 완료되었습니다. 아래에서 결과를 확인하세요.")

else:
    print("❌ 스냅 카드의 링크를 추출하지 못했습니다. 수동 필터 클릭 및 화면 리로드를 확인하세요.")

# --- 5. 수집 결과 데이터프레임 변환 및 출력 ---
if collected_brand_data:
    df_brands = pd.DataFrame(collected_brand_data)
    print(f"\n총 {len(df_brands)}건의 브랜드 및 카테고리 아이템 데이터 적재 완료")
    display(df_brands)
else:
    print("\n❌ 수집된 데이터가 없습니다. HTML 구조나 로딩 대기 시간을 점검해 보세요.")

🚀 현재 화면에 노출된 스냅 카드의 상세 페이지 링크 수집을 시작합니다.
✅ 총 21개의 스냅 상세 페이지 링크를 발견했습니다.

🎯 상위 10개 스냅 페이지 진입 및 하위 상품 카테고리 데이터 수집을 시작합니다.

📸 [스냅 이동 1/10] https://www.musinsa.com/snap/1514148237458964228
-> 태그된 상품 개수: 12개 발견
   🛒 [상품 진입] ID: 6495853 -> https://www.musinsa.com/products/6495853
      ㄴ 🏷️ 카테고리 추출 완료: 상의 | 브랜드: 로우클래식
   🛒 [상품 진입] ID: 4360991 -> https://www.musinsa.com/products/4360991
      ㄴ 🏷️ 카테고리 추출 완료: 바지 | 브랜드: 틸아이다이
   🛒 [상품 진입] ID: 3094203 -> https://www.musinsa.com/products/3094203
      ㄴ 🏷️ 카테고리 추출 완료: 신발 | 브랜드: 사뿐
   🛒 [상품 진입] ID: 5957030 -> https://www.musinsa.com/products/5957030
      ㄴ 🏷️ 카테고리 추출 완료: 가방 | 브랜드: 마조네
   🛒 [상품 진입] ID: 6459551 -> https://www.musinsa.com/products/6459551


KeyboardInterrupt: 